In [1]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
import pandas as pd
import numpy as np

file_name = "PSCompPars_2025.12.02_03.24.48.csv"

print(f"Loading '{file_name}'...")

try:
    all_planets_data = pd.read_csv(file_name, comment='#')
    print(f"Successfully loaded {file_name}.")
    print(f"Original shape (one row per planet): {all_planets_data.shape}")

    mw_data_positives = all_planets_data.drop_duplicates(subset=['hostname'])
    print(f"New shape (one row per STAR): {mw_data_positives.shape}")

    print("\nColumn info:")
    mw_data_positives.info()

    print("\nPreview:")
    print(mw_data_positives.head())

except Exception as e:
    print("Error:", e)

Loading 'PSCompPars_2025.12.02_03.24.48.csv'...
Successfully loaded PSCompPars_2025.12.02_03.24.48.csv.
Original shape (one row per planet): (6052, 24)
New shape (one row per STAR): (4516, 24)

Column info:
<class 'pandas.core.frame.DataFrame'>
Index: 4516 entries, 0 to 6051
Data columns (total 24 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loc_rowid    4516 non-null   int64  
 1   hostname     4516 non-null   object 
 2   sy_snum      4516 non-null   int64  
 3   sy_pnum      4516 non-null   int64  
 4   st_teff      4250 non-null   float64
 5   st_tefferr1  4095 non-null   float64
 6   st_tefferr2  4095 non-null   float64
 7   st_tefflim   4250 non-null   float64
 8   st_rad       4235 non-null   float64
 9   st_raderr1   4118 non-null   float64
 10  st_raderr2   4117 non-null   float64
 11  st_radlim    4235 non-null   float64
 12  st_mass      4510 non-null   float64
 13  st_masserr1  4318 non-null   float64
 14  st_masserr2  4

In [3]:
df_pos = mw_data_positives[[
    "hostname",
    "sy_snum",
    "sy_pnum",
    "st_teff",
    "st_mass",
    "st_rad",
    "st_met"
]]

In [4]:
df_pos_clean = df_pos.dropna(subset=["st_teff", "st_mass", "st_rad", "st_met"])
print("Cleaned dataset shape:", df_pos_clean.shape)
df_pos_clean.head()

Cleaned dataset shape: (4036, 7)


,hostname,sy_snum,sy_pnum,st_teff,st_mass,st_rad,st_met
0,11 Com,2,1,4874.0,2.09,13.76,-0.2600
1,11 UMi,1,1,4213.0,2.78,29.79,-0.0200
2,14 And,1,1,4888.0,1.78,11.55,-0.2100
3,14 Her,1,2,5338.0,0.91,0.93,0.4052
4,16 Cyg B,3,1,5750.0,1.08,1.13,0.0600


In [5]:
def base_planets_from_teff(teff):
    if teff < 3900:  # M dwarfs
        return 4.5
    elif teff < 5200:  # K dwarfs
        return 3.5
    elif teff < 6000:  # G dwarfs
        return 2.5
    elif teff < 7300:  # F dwarfs
        return 1.8
    else:  # A stars
        return 1.0

df_pos_clean["true_planets"] = (
    df_pos_clean["st_teff"].apply(base_planets_from_teff)
)
df_pos_clean.head()

/tmp/ipykernel_2446/946447583.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pos_clean["true_planets"] = (


,hostname,sy_snum,sy_pnum,st_teff,st_mass,st_rad,st_met,true_planets
0,11 Com,2,1,4874.0,2.09,13.76,-0.2600,3.5
1,11 UMi,1,1,4213.0,2.78,29.79,-0.0200,3.5
2,14 And,1,1,4888.0,1.78,11.55,-0.2100,3.5
3,14 Her,1,2,5338.0,0.91,0.93,0.4052,2.5
4,16 Cyg B,3,1,5750.0,1.08,1.13,0.0600,2.5


In [6]:
# Metallicity correction
df_pos_clean.loc[:, "metallicity_factor"] = 10 ** (0.5 * df_pos_clean["st_met"])

# Mass correction 
df_pos_clean.loc[:, "mass_factor"] = df_pos_clean["st_mass"] ** 0.15

# New estimate
df_pos_clean.loc[:, "true_planets_corrected"] = (
    df_pos_clean["true_planets"] * 
    df_pos_clean["metallicity_factor"] * 
    df_pos_clean["mass_factor"]
)

# Reasonable Values
df_pos_clean.loc[:, "true_planets_corrected"] = df_pos_clean["true_planets_corrected"].clip(0.5, 10)

# Result
df_pos_clean.head()

/tmp/ipykernel_2446/3354045698.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pos_clean.loc[:, "metallicity_factor"] = 10 ** (0.5 * df_pos_clean["st_met"])
/tmp/ipykernel_2446/3354045698.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pos_clean.loc[:, "mass_factor"] = df_pos_clean["st_mass"] ** 0.15
/tmp/ipykernel_2446/3354045698.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the ca

,hostname,sy_snum,sy_pnum,st_teff,st_mass,st_rad,st_met,true_planets,metallicity_factor,mass_factor,true_planets_corrected
0,11 Com,2,1,4874.0,2.09,13.76,-0.2600,3.5,0.741310,1.116920,2.897944
1,11 UMi,1,1,4213.0,2.78,29.79,-0.0200,3.5,0.977237,1.165753,3.987262
2,14 And,1,1,4888.0,1.78,11.55,-0.2100,3.5,0.785236,1.090343,2.996616
3,14 Her,1,2,5338.0,0.91,0.93,0.4052,2.5,1.594410,0.985953,3.930033
4,16 Cyg B,3,1,5750.0,1.08,1.13,0.0600,2.5,1.071519,1.011611,2.709902


In [7]:
def classify_star(teff):
    if teff < 3900:
        return "M"
    elif teff < 5200:
        return "K"
    elif teff < 6000:
        return "G"
    elif teff < 7300:
        return "F"
    else:
        return "A"

df_pos_clean.loc[:, "star_type"] = df_pos_clean["st_teff"].apply(classify_star)

df_pos_clean.head()

/tmp/ipykernel_2446/808416337.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pos_clean.loc[:, "star_type"] = df_pos_clean["st_teff"].apply(classify_star)


,hostname,sy_snum,sy_pnum,st_teff,st_mass,st_rad,st_met,true_planets,metallicity_factor,mass_factor,true_planets_corrected,star_type
0,11 Com,2,1,4874.0,2.09,13.76,-0.2600,3.5,0.741310,1.116920,2.897944,K
1,11 UMi,1,1,4213.0,2.78,29.79,-0.0200,3.5,0.977237,1.165753,3.987262,K
2,14 And,1,1,4888.0,1.78,11.55,-0.2100,3.5,0.785236,1.090343,2.996616,K
3,14 Her,1,2,5338.0,0.91,0.93,0.4052,2.5,1.594410,0.985953,3.930033,G
4,16 Cyg B,3,1,5750.0,1.08,1.13,0.0600,2.5,1.071519,1.011611,2.709902,G


In [8]:
# Input features (X)
X = df_pos_clean[["st_teff", "st_mass", "st_rad", "st_met"]]

# Target variable (y)
y = df_pos_clean["true_planets_corrected"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

X.head(), y.head()

Feature matrix shape: (4036, 4)
Target vector shape: (4036,)


(   st_teff  st_mass  st_rad  st_met
 0   4874.0     2.09   13.76 -0.2600
 1   4213.0     2.78   29.79 -0.0200
 2   4888.0     1.78   11.55 -0.2100
 3   5338.0     0.91    0.93  0.4052
 4   5750.0     1.08    1.13  0.0600,
 0    2.897944
 1    3.987262
 2    2.996616
 3    3.930033
 4    2.709902
 Name: true_planets_corrected, dtype: float64)

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (3228, 4)
Test set: (808, 4)


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")

Scaling complete.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Train model
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred_rf)
r2 = r2_score(y_test, y_pred_rf)

print("Random Forest Results:")
print("MSE:", mse)
print("R² Score:", r2)

Random Forest Results:
MSE: 0.00781545213291733
R² Score: 0.9890113005110388


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Define neural network
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)

# Train with scaled data
mlp.fit(X_train_scaled, y_train)

# Predict
y_pred_mlp = mlp.predict(X_test_scaled)

# Evaluate
mse_mlp = mean_squared_error(y_test, y_pred_mlp)
r2_mlp = r2_score(y_test, y_pred_mlp)

print("MLP Neural Network Results:")
print("MSE:", mse_mlp)
print("R² Score:", r2_mlp)

MLP Neural Network Results:
MSE: 0.019978743264041005
R² Score: 0.9719094427088886


In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

N = 10000   # Sample size for GitHub

# Temperature distribution slightly cooler, older stars
# (Most M31 stars are K/G type)
teff = np.random.normal(4800, 700, N).clip(2600, 9000)

# Mass distribution older population means lower mass on average
mass = np.random.normal(0.95, 0.25, N).clip(0.1, 2.5)

# Radius correlated with mass but include some noise
radius = (mass ** 0.8) * np.random.normal(1.0, 0.2, N)
radius = radius.clip(0.1, 10)

# Metallicity Andromeda is slightly more metal-rich on average
metallicity = np.random.normal(0.0, 0.25, N).clip(-1.0, 0.5)

andromeda_synth = pd.DataFrame({
    "st_teff": teff,
    "st_mass": mass,
    "st_rad": radius,
    "st_met": metallicity
})

andromeda_synth.to_csv("andromeda_synthetic.csv", index=False)

andromeda_synth.describe()
print("Synthetic Andromeda dataset created and saved as 'andromeda_synthetic.csv'.")

Synthetic Andromeda dataset created and saved as 'andromeda_synthetic.csv'.


In [17]:
andromeda = pd.read_csv("andromeda_synthetic.csv")
andromeda.head()

,st_teff,st_mass,st_rad,st_met
0,5147.699907,0.780376,0.877178,-0.495143
1,4703.214989,0.873625,0.948413,-0.263746
2,5253.381977,0.800655,0.680275,-0.146757
3,5866.120899,0.977605,1.095878,0.037417
4,4636.092638,1.249295,0.838801,0.256041


In [18]:
X_and = andromeda[["st_teff", "st_mass", "st_rad", "st_met"]]
X_and.head()

,st_teff,st_mass,st_rad,st_met
0,5147.699907,0.780376,0.877178,-0.495143
1,4703.214989,0.873625,0.948413,-0.263746
2,5253.381977,0.800655,0.680275,-0.146757
3,5866.120899,0.977605,1.095878,0.037417
4,4636.092638,1.249295,0.838801,0.256041


In [19]:
X_and_scaled = scaler.transform(X_and)

In [20]:
andromeda["predicted_planets_rf"] = rf.predict(X_and)
andromeda[["predicted_planets_rf"]].head()

,predicted_planets_rf
0,1.926757
1,2.563661
2,2.056291
3,2.605421
4,4.689912


In [21]:
andromeda["predicted_planets_mlp"] = mlp.predict(X_and_scaled)
andromeda[["predicted_planets_mlp"]].head()

,predicted_planets_mlp
0,1.605012
1,2.443266
2,2.035096
3,2.602463
4,5.070814


In [22]:
andromeda[["predicted_planets_rf", "predicted_planets_mlp"]].mean()

predicted_planets_rf     3.320777
predicted_planets_mlp    3.351599
dtype: float64

In [23]:
avg_planets = andromeda["predicted_planets_rf"].mean()
total_planets = avg_planets * 1e12
total_planets

np.float64(3320776825904.002)

In [24]:
avg_planets_mlp = andromeda["predicted_planets_mlp"].mean()
total_planets_mlp = avg_planets_mlp * 1e12
total_planets_mlp

np.float64(3351599414623.1123)

In [ ]:
import numpy as np
import pandas as pd

# Total planet estimates from ML models 
avg_rf = andromeda["predicted_planets_rf"].mean()
avg_mlp = andromeda["predicted_planets_mlp"].mean()

# Accepted estimate: ~1 trillion stars in Andromeda
num_stars_andromeda = 1e12

total_planets_rf = avg_rf * num_stars_andromeda
total_planets_mlp = avg_mlp * num_stars_andromeda

# Astrophysical assumptions for habitability 
rocky_fraction = 0.40      # ~40% of planets are rocky (Kepler results)
hz_fraction = 0.25         # ~25% of rocky planets are in the habitable zone

# Compute rocky and HZ planets 
rocky_planets_rf = total_planets_rf * rocky_fraction
hz_planets_rf = rocky_planets_rf * hz_fraction

rocky_planets_mlp = total_planets_mlp * rocky_fraction
hz_planets_mlp = rocky_planets_mlp * hz_fraction

# Display results 
results = pd.DataFrame({
    "Model": ["Random Forest", "MLP Neural Network"],
    "Avg Planets per Star": [avg_rf, avg_mlp],
    "Total Planets (Trillions)": [total_planets_rf/1e12, total_planets_mlp/1e12],
    "Rocky Planets (Trillions)": [rocky_planets_rf/1e12, rocky_planets_mlp/1e12],
    "Habitable-Zone Planets (Billions)": [hz_planets_rf/1e9, hz_planets_mlp/1e9]
})

results

,Model,Avg Planets per Star,Total Planets (Trillions),Rocky Planets (Trillions),Habitable-Zone Planets (Billions)
0,Random Forest,3.320777,3.320777,1.328311,332.077683
1,MLP Neural Network,3.351599,3.351599,1.340640,335.159941


In [ ]:
# AI Disclosure: I consulted and used GPT 5 and Gemini for astrophysics/project advice, help, and error code issues. 